# KLUE-RoBERTa — 전체 데이터 최종 학습

5-Fold CV에서 최고 성능 확정 → work_pool 전체(100%)로 재학습 → test_final 평가용 모델 저장

| 항목 | 값 |
|------|----|)
| 모델 | `klue/roberta-base` |
| 데이터 | work_pool 전체 (중복 제거 후 ~291,215건) |
| 저장 | `klue_binary_final.pt`, `klue_multi_final.pt` |

## Step 0. 패키지 설치

In [ ]:
!pip install -q transformers torch sentencepiece protobuf scikit-learn tqdm

## Step 1. Google Drive 마운트 + 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DATA_DIR = '/content/drive/MyDrive/text-mining-2026/data/processed'
SAVE_DIR = '/content/drive/MyDrive/text-mining-2026/models'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'데이터 경로: {DATA_DIR}')
print(f'저장 경로:   {SAVE_DIR}')

## Step 2. 라이브러리 임포트

In [ ]:
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 3. 데이터 로딩 + 중복 제거

In [ ]:
df = pd.concat([
    pd.read_parquet(f'{DATA_DIR}/work_pool_clickbait_auto.parquet'),
    pd.read_parquet(f'{DATA_DIR}/work_pool_clickbait_direct.parquet'),
    pd.read_parquet(f'{DATA_DIR}/work_pool_nonclickbait_auto.parquet'),
], ignore_index=True)

before = len(df)
df = df.drop_duplicates(subset=['title_clean', 'content_clean']).reset_index(drop=True)
print(f'중복 제거: {before:,} → {len(df):,}건 ({before - len(df)}건 제거)')

df_multi = df[df['type_label'] != -1]
print(f'이진 분류: {len(df):,}건')
print(f'다중 분류: {len(df_multi):,}건 (Clickbait_Direct)')

## Step 4. 설정값

In [ ]:
MODEL_NAME = 'klue/roberta-base'
MODEL_KEY  = 'klue'

BATCH_SIZE    = 64
MAX_LENGTH    = 512
LR            = 2e-5
WARMUP_RATIO  = 0.1
MAX_GRAD_NORM = 1.0
WEIGHT_DECAY  = 0.01
# 검증셋 없으므로 early stopping 없이 고정 epoch 학습
EPOCHS = {'binary': 3, 'multi': 5}

print(f'모델: {MODEL_NAME}')
print(f'BATCH={BATCH_SIZE}, LR={LR}, EPOCHS={EPOCHS}')

## Step 5. Dataset / 헬퍼 함수

In [ ]:
class ClickbaitDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=MAX_LENGTH, task='binary'):
        if task == 'multi':
            dataframe = dataframe[dataframe['type_label'] != -1].reset_index(drop=True)
        self.titles   = dataframe['title_clean'].tolist()
        self.contents = dataframe['content_clean'].tolist()
        self.labels   = dataframe[
            'binary_label' if task == 'binary' else 'type_label'
        ].tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            text=self.titles[idx],
            text_pair=self.contents[idx],
            truncation='only_second',
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        item = {
            'input_ids':      encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
        }
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def get_class_weights(labels, device='cpu'):
    classes = np.array(sorted(set(labels)))
    weights = compute_class_weight('balanced', classes=classes, y=np.array(labels))
    return torch.tensor(weights, dtype=torch.float).to(device)

print('Dataset / 헬퍼 함수 정의 완료')

## Step 6. 학습 함수

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []
    for batch in tqdm(loader, desc='  train', leave=False):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        optimizer.zero_grad()
        with torch.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        all_preds.extend(torch.argmax(outputs.logits, dim=-1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    train_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return total_loss / len(loader), train_f1

print('train_epoch 정의 완료')

## Step 7. 전체 데이터 학습 함수

In [ ]:
def run_full_train(df, task='binary'):
    num_labels = 2 if task == 'binary' else 6
    n_epochs   = EPOCHS[task]
    save_path  = f'{SAVE_DIR}/{MODEL_KEY}_{task}_final.pt'

    if os.path.exists(save_path):
        print(f'[{MODEL_KEY}/{task}] 이미 완료 → 스킵')
        return

    print(f"\n{'='*60}")
    print(f'  KLUE-RoBERTa | {task} | 전체 데이터 재학습')
    print(f"{'='*60}")

    tok     = AutoTokenizer.from_pretrained(MODEL_NAME)
    dataset = ClickbaitDataset(df, tok, task=task)
    loader  = DataLoader(
        dataset, batch_size=BATCH_SIZE,
        shuffle=True, num_workers=2, pin_memory=True,
    )
    print(f'  Dataset: {len(dataset):,}건 | Steps/epoch: {len(loader):,}')

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=num_labels,
    ).to(device)

    if task == 'binary':
        criterion = nn.CrossEntropyLoss()
    else:
        weights = get_class_weights(dataset.labels, device=device)
        criterion = nn.CrossEntropyLoss(weight=weights)
        print(f'  class weights: {[f"{w:.3f}" for w in weights.cpu().numpy()]}')

    total_steps  = len(loader) * n_epochs
    warmup_steps = int(total_steps * WARMUP_RATIO)
    optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler    = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps,
    )
    print(f'  총 스텝: {total_steps:,} | warmup: {warmup_steps:,}')

    for epoch in range(n_epochs):
        t0 = time.time()
        train_loss, train_f1 = train_epoch(model, loader, optimizer, scheduler, criterion)
        elapsed = time.time() - t0
        print(f'  Epoch {epoch+1}/{n_epochs} ({elapsed:.0f}s) | '
              f'loss={train_loss:.4f} | train_f1={train_f1:.4f}')

    torch.save(model.state_dict(), save_path)
    print(f'  ✅ 저장 완료: {save_path}')

    del model
    torch.cuda.empty_cache()

print('run_full_train 정의 완료')

## Step 8. 이진 분류 전체 학습

In [ ]:
run_full_train(df, task='binary')

## Step 9. 다중 분류 전체 학습

In [ ]:
run_full_train(df, task='multi')

## Step 10. 저장 확인

In [ ]:
print('최종 저장 파일:')
for task in ['binary', 'multi']:
    path = f'{SAVE_DIR}/{MODEL_KEY}_{task}_final.pt'
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'  ✅ {os.path.basename(path)} ({size_mb:.0f}MB)')
    else:
        print(f'  ❌ {os.path.basename(path)} — 생성 실패')